# Austrian electricity system

Live ENTSO-E data normalized by GridScope. Configure `ENTSOE_API_TOKEN` in the environment or project `.env`. All dates below are UTC. No data are embedded in this notebook.

Start Jupyter from the repository root: `uv run --group examples jupyter lab`. Execute cells in order.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from gridscope import GridScope
from gridscope.intelligence.metrics import renewable_share, residual_load

start, end = "2026-09-01", "2026-09-02"
async with GridScope() as gs:
    prices = await gs.prices("AT", start, end)
    load = await gs.load("AT", start, end)
    generation = await gs.generation("AT", start, end)

prices.to_dataframe().head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
load.to_dataframe()["value"].plot(ax=axes[0], title="Austrian actual load", ylabel="MW")
generation.to_dataframe().reset_index().pivot(
    index="timestamp", columns="generation_type", values="value"
).plot(ax=axes[1], ylabel="MW", title="Reported generation by type")
prices.to_dataframe()["value"].plot(
    ax=axes[2], ylabel="EUR/MWh", title="Day-ahead prices", color="darkorange"
)
axes[2].set_xlabel("UTC")
fig.suptitle("Source: ENTSO-E Transparency Platform · normalized by GridScope")
fig.tight_layout()

In [ ]:
shares = renewable_share(generation.data)
residual = residual_load(load.data, generation.data)
pd.DataFrame([p.model_dump() for p in shares]).set_index("timestamp")[["value", "quality"]].head()

Renewable share uses reported generation only. Pumped storage and mixed waste are excluded from the renewable numerator. Residual load subtracts reported solar and wind. Missing values remain missing. Cross-series metrics require matching resolutions and never silently interpolate. Check `series.meta.warnings` before interpreting coverage.

In [ ]:
for series in (load, generation, prices):
    print(series.meta.dataset, series.meta.warnings)